# LeetCode 1570 — Dot Product of Two Sparse Vectors bằng Python

Link đề: https://leetcode.com/problems/dot-product-of-two-sparse-vectors/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Two Pointers, Design, Linear Algebra

> **Ghi chú Premium:** đây là bài Premium (🔒). Đề bài và chữ ký dưới đây được viết theo hiểu biết về bài, lấy từ trang lời giải doocs (https://leetcode.doocs.org/en/lc/1570/). Nếu bạn có tài khoản Premium, hãy đối chiếu lại với đề gốc trên LeetCode.

Skeleton LeetCode (bài **thiết kế lớp**):

```python
class SparseVector:
    def __init__(self, nums: List[int]):
        ...

    # Return the dotProduct of two sparse vectors
    def dotProduct(self, vec: "SparseVector") -> int:
        ...

# Your SparseVector object will be instantiated and called as such:
# v1 = SparseVector(nums1)
# v2 = SparseVector(nums2)
# ans = v1.dotProduct(v2)
```

## 1. Mục tiêu học

1. Hiểu **tích vô hướng** (dot product) của hai vector và vì sao với vector **thưa** (đa số phần tử bằng 0) ta chỉ cần quan tâm các vị trí khác 0.
2. Biết cách **lưu trữ vector thưa** bằng ba kiểu: mảng đầy đủ (baseline), **dict chỉ số → giá trị**, và **danh sách cặp `(chỉ số, giá trị)` đã sắp theo chỉ số**.
3. Cài đặt tích vô hướng bằng **tra dict** (duyệt dict nhỏ hơn) và bằng **hai con trỏ** trên hai danh sách cặp.
4. Trả lời câu **follow-up**: nếu chỉ một vector thưa (vector kia dày) thì nên duyệt bên nào, dùng tra chỉ số trực tiếp hay tìm kiếm nhị phân.
5. Viết **bộ chạy chuỗi lệnh giả lập** cho bài thiết kế lớp và kiểm thử ngẫu nhiên với seed cố định.

## 2. Đề bài & ví dụ

Cho hai vector **thưa** (sparse vector) có cùng độ dài `n`, hãy tính **tích vô hướng** của chúng.

Cài đặt lớp `SparseVector`:

- `SparseVector(nums)`: khởi tạo đối tượng từ mảng `nums`.
- `dotProduct(vec)`: trả về tích vô hướng giữa vector hiện tại (`self`) và vector `vec` (cũng là một `SparseVector`).

"Vector thưa" là vector có **hầu hết phần tử bằng 0**. Bạn nên **lưu trữ hiệu quả** vector thưa và tính tích vô hướng giữa hai `SparseVector`.

**Follow-up:** Nếu chỉ **một** trong hai vector là thưa thì sao?

**Ví dụ 1:** `nums1 = [1,0,0,2,3]`, `nums2 = [0,3,0,4,0]` → `8`.
`1*0 + 0*3 + 0*0 + 2*4 + 3*0 = 8`.

**Ví dụ 2:** `nums1 = [0,1,0,0,0]`, `nums2 = [0,0,0,0,2]` → `0`.

**Ví dụ 3:** `nums1 = [0,1,0,0,2,0,0]`, `nums2 = [1,0,0,0,3,0,4]` → `6` (chỉ vị trí 4 cùng khác 0: `2*3 = 6`).

**Ràng buộc:**
- `n == nums1.length == nums2.length`
- `1 ≤ n ≤ 10⁵`
- `0 ≤ nums1[i], nums2[i] ≤ 100`

Trên LeetCode, đầu vào là `nums1`, `nums2`; hệ thống sẽ gọi `v1 = SparseVector(nums1)`, `v2 = SparseVector(nums2)`, rồi `v1.dotProduct(v2)`.

In [1]:
import random
import time
from bisect import bisect_left
from typing import List

examples = [
    ([1, 0, 0, 2, 3], [0, 3, 0, 4, 0], 8),
    ([0, 1, 0, 0, 0], [0, 0, 0, 0, 2], 0),
    ([0, 1, 0, 0, 2, 0, 0], [1, 0, 0, 0, 3, 0, 4], 6),
]
for a, b, want in examples:
    direct = sum(x * y for x, y in zip(a, b))
    print(f"nums1={a}, nums2={b} -> tính thẳng {direct}, mong đợi {want}")

nums1=[1, 0, 0, 2, 3], nums2=[0, 3, 0, 4, 0] -> tính thẳng 8, mong đợi 8
nums1=[0, 1, 0, 0, 0], nums2=[0, 0, 0, 0, 2] -> tính thẳng 0, mong đợi 0
nums1=[0, 1, 0, 0, 2, 0, 0], nums2=[1, 0, 0, 0, 3, 0, 4] -> tính thẳng 6, mong đợi 6


## 3. Kiến thức nền

### 3.1 Tích vô hướng là gì?

Với hai vector cùng độ dài `a = (a₀, …, a_{n-1})` và `b = (b₀, …, b_{n-1})`:

`a · b = a₀·b₀ + a₁·b₁ + … + a_{n-1}·b_{n-1}`

Mỗi vị trí `i` đóng góp `a[i] * b[i]`. Nếu **một trong hai** bằng 0 thì đóng góp bằng 0 → có thể bỏ qua.

In [2]:
a = [1, 0, 0, 2, 3]
b = [0, 3, 0, 4, 0]
for i in range(len(a)):
    print(f"i={i}: {a[i]} * {b[i]} = {a[i] * b[i]}")
print("tổng =", sum(a[i] * b[i] for i in range(len(a))))

i=0: 1 * 0 = 0
i=1: 0 * 3 = 0
i=2: 0 * 0 = 0
i=3: 2 * 4 = 8
i=4: 3 * 0 = 0
tổng = 8


### 3.2 `zip` và `enumerate`

- `zip(a, b)` ghép từng cặp phần tử cùng vị trí.
- `enumerate(nums)` trả về cặp `(chỉ số, giá trị)` — rất tiện để lấy ra các vị trí khác 0.

In [3]:
nums = [0, 7, 0, 0, 5]
print(list(zip([1, 2, 3], [4, 5, 6])))
print(list(enumerate(nums)))
print([(i, v) for i, v in enumerate(nums) if v != 0])

[(1, 4), (2, 5), (3, 6)]
[(0, 0), (1, 7), (2, 0), (3, 0), (4, 5)]
[(1, 7), (4, 5)]


### 3.3 Dict comprehension và `dict.get`

- `{i: v for i, v in enumerate(nums) if v}` tạo dict chỉ chứa các vị trí khác 0 (`if v` đúng khi `v != 0`).
- `d.get(key, 0)` trả về giá trị nếu có khoá, ngược lại trả `0` — **không** ném `KeyError` như `d[key]`.

In [4]:
d = {i: v for i, v in enumerate([1, 0, 0, 2, 3]) if v}
print(d)
print("d.get(3, 0) =", d.get(3, 0), "| d.get(1, 0) =", d.get(1, 0))
try:
    print(d[1])
except KeyError as e:
    print("d[1] ném KeyError:", e)

{0: 1, 3: 2, 4: 3}
d.get(3, 0) = 2 | d.get(1, 0) = 0
d[1] ném KeyError: 1


### 3.4 Lớp (class) và phương thức nhận một đối tượng **cùng lớp**

Trong `dotProduct(self, vec)`, `vec` là một `SparseVector` khác. Ta được phép đọc thuộc tính của nó (`vec.d`, `vec.pairs`, …). Chú thích kiểu `"SparseVector"` viết trong dấu nháy vì lúc định nghĩa phương thức, tên lớp chưa tồn tại xong (forward reference).

In [5]:
class Point:
    def __init__(self, x, y):
        self.x, self.y = x, y

    def dot(self, other: "Point") -> int:
        return self.x * other.x + self.y * other.y

p, q = Point(1, 2), Point(3, 4)
print("p·q =", p.dot(q), "| q·p =", q.dot(p))

p·q = 11 | q·p = 11


### 3.5 Generator trong `sum(...)` và hoán đổi biến

- `sum(v * b.get(i, 0) for i, v in a.items())` cộng dồn mà không tạo list trung gian.
- `a, b = b, a` hoán đổi hai biến trong một dòng — dùng để luôn duyệt dict **nhỏ hơn**.

In [6]:
a = {0: 1, 3: 2, 4: 3}
b = {1: 3, 3: 4}
if len(b) < len(a):
    a, b = b, a
print("duyệt dict có", len(a), "phần tử:", a)
print("tích =", sum(v * b.get(i, 0) for i, v in a.items()))

duyệt dict có 2 phần tử: {1: 3, 3: 4}
tích = 8


### 3.6 Tìm kiếm nhị phân với `bisect_left`

`bisect_left(arr, x)` trả về vị trí đầu tiên trong mảng **đã sắp** mà `arr[pos] >= x`. Kiểm tra `pos < len(arr) and arr[pos] == x` để biết `x` có trong mảng hay không. Ta sẽ dùng nó trong follow-up khi một vector rất thưa, vector kia ít thưa hơn.

In [7]:
idx = [0, 3, 4, 9, 15]
for x in [3, 5, 15, 20]:
    pos = bisect_left(idx, x)
    found = pos < len(idx) and idx[pos] == x
    print(f"x={x}: pos={pos}, có trong mảng? {found}")

x=3: pos=1, có trong mảng? True
x=5: pos=3, có trong mảng? False
x=15: pos=4, có trong mảng? True
x=20: pos=5, có trong mảng? False


## 4. Ý tưởng nền tảng / trực giác

Tính tay ví dụ 1: `nums1 = [1,0,0,2,3]`, `nums2 = [0,3,0,4,0]`.

| i | nums1[i] | nums2[i] | tích | ghi chú |
|---|---|---|---|---|
| 0 | 1 | 0 | 0 | nums2 bằng 0 |
| 1 | 0 | 3 | 0 | nums1 bằng 0 |
| 2 | 0 | 0 | 0 | cả hai bằng 0 |
| 3 | 2 | 4 | **8** | cả hai khác 0 |
| 4 | 3 | 0 | 0 | nums2 bằng 0 |

Quan sát:
- Vị trí khác 0 của `nums1`: `{0, 3, 4}`; của `nums2`: `{1, 3}`.
- Chỉ **giao** `{3}` đóng góp vào tổng.

⇒ Nếu ta chỉ lưu các vị trí khác 0, phép nhân chỉ cần xét **giao** của hai tập chỉ số. Với vector thưa (ví dụ `n = 10⁵` nhưng chỉ ~100 phần tử khác 0), số phép tính giảm khoảng 1000 lần.

In [8]:
nums1, nums2 = [1, 0, 0, 2, 3], [0, 3, 0, 4, 0]
nz1 = {i for i, v in enumerate(nums1) if v}
nz2 = {i for i, v in enumerate(nums2) if v}
common = nz1 & nz2
print("khác 0 ở nums1:", nz1, "| nums2:", nz2, "| giao:", common)
print("tích =", sum(nums1[i] * nums2[i] for i in common))

khác 0 ở nums1: {0, 3, 4} | nums2: {1, 3} | giao: {3}
tích = 8


## 5. Cách 1 — Brute Force / baseline: lưu nguyên mảng

**Ý tưởng:** `__init__` giữ lại nguyên mảng `nums`; `dotProduct` duyệt toàn bộ `n` vị trí và cộng `a[i] * b[i]`.

Đây là cách "đúng định nghĩa", luôn đúng, dùng làm **chuẩn đối chiếu** (oracle) cho các cách sau.

In [9]:
class SparseVectorDense:
    """Baseline: lưu mảng đầy đủ, nhân trực tiếp O(n)."""

    def __init__(self, nums: List[int]):
        self.nums = list(nums)

    def dotProduct(self, vec: "SparseVectorDense") -> int:
        total = 0
        for x, y in zip(self.nums, vec.nums):
            total += x * y
        return total

for a, b, want in examples:
    got = SparseVectorDense(a).dotProduct(SparseVectorDense(b))
    print(got, "| mong đợi", want, "| OK" if got == want else "| SAI")

8 | mong đợi 8 | OK
0 | mong đợi 0 | OK
6 | mong đợi 6 | OK


**Độ phức tạp:** khởi tạo `O(n)` thời gian và bộ nhớ; `dotProduct` luôn `O(n)` — kể cả khi chỉ có 1–2 phần tử khác 0.

**Nút thắt:**
- Lãng phí bộ nhớ: lưu `10⁵` số mà phần lớn là 0.
- Lãng phí thời gian: nếu cùng một vector được nhân với rất nhiều vector khác (ví dụ tính độ tương đồng văn bản), mỗi lần đều tốn `O(n)`.

Đề yêu cầu "lưu trữ hiệu quả" → baseline này chấp nhận được về đáp án nhưng không đúng tinh thần bài.

In [10]:
# Đo nhanh: n = 10^5, mỗi vector chỉ ~100 phần tử khác 0
rng = random.Random(1570)
n = 10**5
def random_sparse(n, k, rng):
    arr = [0] * n
    for i in rng.sample(range(n), k):
        arr[i] = rng.randint(1, 100)
    return arr

big1, big2 = random_sparse(n, 100, rng), random_sparse(n, 100, rng)
t0 = time.perf_counter()
for _ in range(20):
    SparseVectorDense(big1).dotProduct(SparseVectorDense(big2))
print(f"Baseline 20 lần: {time.perf_counter() - t0:.3f}s")

Baseline 20 lần: 0.077s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** `a[i] * b[i] ≠ 0` **khi và chỉ khi** `a[i] ≠ 0` **và** `b[i] ≠ 0`. Vậy:

`a · b = Σ_{i ∈ NZ(a) ∩ NZ(b)} a[i] · b[i]` với `NZ(x)` = tập chỉ số khác 0 của `x`.

Từ đó có các quy tắc:

- **Nếu** ta lưu `NZ(a)` dưới dạng **dict** (tra cứu `O(1)`) **thì** duyệt từng phần tử của dict **nhỏ hơn** và tra bên kia → `O(min(k₁, k₂))`.
- **Nếu** ta lưu dưới dạng **danh sách cặp đã sắp theo chỉ số** **thì** dùng **hai con trỏ** giống trộn (merge) hai mảng sắp xếp → `O(k₁ + k₂)`, không cần hash.
  - chỉ số bằng nhau → cộng tích, tiến cả hai;
  - chỉ số bên trái nhỏ hơn → tiến con trỏ trái (chỉ số đó không thể có bên phải);
  - ngược lại → tiến con trỏ phải.
- **Nếu** một vector rất thưa (`k₁` nhỏ) còn vector kia nhiều phần tử hơn (`k₂` lớn) **thì** với mỗi chỉ số của vector thưa, **tìm nhị phân** trong vector kia → `O(k₁ log k₂)`; nếu vector kia **dày** (lưu mảng đầy) thì tra trực tiếp `b[i]` → `O(k₁)`.

Ở đây `k₁, k₂` = số phần tử khác 0 của hai vector.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Dict chỉ số → giá trị (lời giải chuẩn của doocs)

**Bước 1 — Khởi tạo:** chỉ lưu các cặp `i: v` với `v ≠ 0`.

**Bước 2 — Nhân:** gọi `a` là dict nhỏ hơn, `b` là dict lớn hơn. Với mỗi `(i, v)` trong `a`, cộng `v * b.get(i, 0)`.

Vì sao duyệt dict nhỏ hơn? Mỗi phép tra `b.get` là `O(1)` trung bình, nên tổng chi phí là `O(min(k₁, k₂))`. Nếu duyệt bên lớn thì tốn `O(max(k₁, k₂))` một cách vô ích.

In [11]:
class SparseVectorDict:
    def __init__(self, nums: List[int]):
        self.d = {i: v for i, v in enumerate(nums) if v}

    def dotProduct(self, vec: "SparseVectorDict") -> int:
        a, b = self.d, vec.d
        if len(b) < len(a):
            a, b = b, a
        return sum(v * b.get(i, 0) for i, v in a.items())

v1 = SparseVectorDict([1, 0, 0, 2, 3])
v2 = SparseVectorDict([0, 3, 0, 4, 0])
print("v1.d =", v1.d, "| v2.d =", v2.d)
print("v1·v2 =", v1.dotProduct(v2), "| v2·v1 =", v2.dotProduct(v1))

v1.d = {0: 1, 3: 2, 4: 3} | v2.d = {1: 3, 3: 4}
v1·v2 = 8 | v2·v1 = 8


Code nộp LeetCode cho cách 2 (giống phần 9):

```python
class SparseVector:
    def __init__(self, nums: List[int]):
        self.d = {i: v for i, v in enumerate(nums) if v}

    def dotProduct(self, vec: "SparseVector") -> int:
        a, b = self.d, vec.d
        if len(b) < len(a):
            a, b = b, a
        return sum(v * b.get(i, 0) for i, v in a.items())
```

**Độ phức tạp:** khởi tạo `O(n)` thời gian, `O(k)` bộ nhớ; `dotProduct` `O(min(k₁, k₂))` trung bình.

**Lưu ý:** hash table có hằng số lớn và trong trường hợp xấu (va chạm hash) có thể chậm; một số người phỏng vấn sẽ hỏi tiếp "làm sao không cần hash?" → cách 3.

### 7.2 Cách 3 — Danh sách cặp `(chỉ số, giá trị)` + hai con trỏ

**Bước 1 — Khởi tạo:** `self.pairs = [(i, v) for i, v in enumerate(nums) if v]`. Vì duyệt `i` tăng dần nên danh sách **tự động đã sắp** theo chỉ số.

**Bước 2 — Hai con trỏ `p`, `q`:**
- `i1 == i2` → cộng `v1 * v2`, tăng cả `p`, `q`;
- `i1 < i2` → tăng `p` (chỉ số `i1` không thể xuất hiện ở phía sau của danh sách kia vì danh sách kia đã ở chỉ số lớn hơn);
- `i1 > i2` → tăng `q`.

Dừng khi một trong hai danh sách hết.

In [12]:
class SparseVectorPairs:
    def __init__(self, nums: List[int]):
        self.pairs = [(i, v) for i, v in enumerate(nums) if v]

    def dotProduct(self, vec: "SparseVectorPairs") -> int:
        A, B = self.pairs, vec.pairs
        p = q = 0
        total = 0
        while p < len(A) and q < len(B):
            i1, v1 = A[p]
            i2, v2 = B[q]
            if i1 == i2:
                total += v1 * v2
                p += 1
                q += 1
            elif i1 < i2:
                p += 1
            else:
                q += 1
        return total

for a, b, want in examples:
    got = SparseVectorPairs(a).dotProduct(SparseVectorPairs(b))
    print(SparseVectorPairs(a).pairs, SparseVectorPairs(b).pairs, "->", got, "| mong đợi", want)

[(0, 1), (3, 2), (4, 3)] [(1, 3), (3, 4)] -> 8 | mong đợi 8
[(1, 1)] [(4, 2)] -> 0 | mong đợi 0
[(1, 1), (4, 2)] [(0, 1), (4, 3), (6, 4)] -> 6 | mong đợi 6


Code nộp LeetCode cho cách 3:

```python
class SparseVector:
    def __init__(self, nums: List[int]):
        self.pairs = [(i, v) for i, v in enumerate(nums) if v]

    def dotProduct(self, vec: "SparseVector") -> int:
        A, B = self.pairs, vec.pairs
        p = q = total = 0
        while p < len(A) and q < len(B):
            if A[p][0] == B[q][0]:
                total += A[p][1] * B[q][1]
                p += 1
                q += 1
            elif A[p][0] < B[q][0]:
                p += 1
            else:
                q += 1
        return total
```

**Độ phức tạp:** khởi tạo `O(n)`; `dotProduct` `O(k₁ + k₂)` **xấu nhất** (không phụ thuộc chất lượng hàm hash); bộ nhớ `O(k)`.

Đây thường là câu trả lời mà người phỏng vấn mong đợi ở vòng 2 (ví dụ ở các công ty dùng bài này làm câu hỏi phỏng vấn phổ biến).

### 7.3 Cách 4 — Follow-up: chỉ một vector thưa

Có hai tình huống:

**(a) Vector kia lưu dạng mảng đầy (dày).** Duyệt các cặp `(i, v)` của vector **thưa** và lấy `dense[i]` trực tiếp → `O(k)` với `k` = số phần tử khác 0 của vector thưa. Không cần hash, không cần hai con trỏ.

**(b) Cả hai lưu dạng cặp nhưng kích thước chênh lệch lớn** (`k₁ ≪ k₂`). Hai con trỏ tốn `O(k₁ + k₂)` ≈ `O(k₂)`. Thay vào đó, với mỗi chỉ số của vector nhỏ, **tìm nhị phân** trong danh sách chỉ số của vector lớn → `O(k₁ log k₂)`. Khi `k₁ = 10`, `k₂ = 10⁵` thì `10 · 17 = 170` phép so sánh thay vì `10⁵`.

Ta cài một lớp "lai": lưu cả danh sách chỉ số và danh sách giá trị, tự chọn chiến lược.

In [13]:
class SparseVectorHybrid:
    def __init__(self, nums: List[int]):
        self.idx = [i for i, v in enumerate(nums) if v]
        self.val = [nums[i] for i in self.idx]

    def dot_with_dense(self, dense: List[int]) -> int:
        """Follow-up (a): vector kia dày, tra trực tiếp theo chỉ số."""
        return sum(v * dense[i] for i, v in zip(self.idx, self.val))

    def dotProduct(self, vec: "SparseVectorHybrid") -> int:
        """Follow-up (b): duyệt bên ít phần tử, tìm nhị phân bên nhiều phần tử."""
        small, big = (self, vec) if len(self.idx) <= len(vec.idx) else (vec, self)
        total = 0
        lo = 0                      # vị trí tìm kiếm có thể tăng dần vì idx đã sắp
        for i, v in zip(small.idx, small.val):
            pos = bisect_left(big.idx, i, lo)
            if pos == len(big.idx):
                break
            if big.idx[pos] == i:
                total += v * big.val[pos]
            lo = pos
        return total

for a, b, want in examples:
    s = SparseVectorHybrid(a)
    print("hybrid:", s.dotProduct(SparseVectorHybrid(b)), "| với mảng dày:", s.dot_with_dense(b), "| mong đợi", want)

hybrid: 8 | với mảng dày: 8 | mong đợi 8
hybrid: 0 | với mảng dày: 0 | mong đợi 0
hybrid: 6 | với mảng dày: 6 | mong đợi 6


Giải thích chi tiết đoạn `bisect_left(big.idx, i, lo)`:
- `big.idx` đã sắp tăng dần nên tìm nhị phân được.
- Các chỉ số `i` của `small` cũng tăng dần, nên vị trí tìm thấy lần sau **không nhỏ hơn** lần trước → truyền `lo = pos` để thu hẹp vùng tìm (vẫn đúng, nhanh hơn chút).
- Nếu `pos == len(big.idx)`: mọi chỉ số còn lại của `big` đều nhỏ hơn `i` → các `i` sau càng lớn hơn → dừng sớm.

Code nộp LeetCode (follow-up, vẫn đúng chữ ký):

```python
class SparseVector:
    def __init__(self, nums: List[int]):
        self.idx = [i for i, v in enumerate(nums) if v]
        self.val = [nums[i] for i in self.idx]

    def dotProduct(self, vec: "SparseVector") -> int:
        small, big = (self, vec) if len(self.idx) <= len(vec.idx) else (vec, self)
        total, lo = 0, 0
        for i, v in zip(small.idx, small.val):
            pos = bisect_left(big.idx, i, lo)
            if pos == len(big.idx):
                break
            if big.idx[pos] == i:
                total += v * big.val[pos]
            lo = pos
        return total
```

**Độ phức tạp:** `O(k_small · log k_big)`; bộ nhớ `O(k)`.

In [14]:
# So sánh tốc độ trên vector thưa lớn (cùng dữ liệu big1, big2 ở trên)
def bench(cls, a, b, reps=200):
    va, vb = cls(a), cls(b)          # khởi tạo 1 lần, nhân nhiều lần
    t0 = time.perf_counter()
    for _ in range(reps):
        r = va.dotProduct(vb)
    return r, time.perf_counter() - t0

for cls in [SparseVectorDict, SparseVectorPairs, SparseVectorHybrid]:
    r, t = bench(cls, big1, big2)
    print(f"{cls.__name__:<20} kết quả={r:<6} 200 lần nhân: {t:.4f}s")
r, t = bench(SparseVectorDense, big1, big2, reps=5)
print(f"{'SparseVectorDense':<20} kết quả={r:<6} chỉ 5 lần nhân: {t:.4f}s")

SparseVectorDict     kết quả=0      200 lần nhân: 0.0014s
SparseVectorPairs    kết quả=0      200 lần nhân: 0.0041s
SparseVectorHybrid   kết quả=0      200 lần nhân: 0.0023s
SparseVectorDense    kết quả=0      chỉ 5 lần nhân: 0.0157s


## 8. TRACE từng bước (hai con trỏ)

Ví dụ 3: `nums1 = [0,1,0,0,2,0,0]` → `A = [(1,1), (4,2)]`; `nums2 = [1,0,0,0,3,0,4]` → `B = [(0,1), (4,3), (6,4)]`.

| bước | p | q | A[p] | B[q] | so sánh | hành động | total |
|---|---|---|---|---|---|---|---|
| 1 | 0 | 0 | (1,1) | (0,1) | 1 > 0 | q += 1 | 0 |
| 2 | 0 | 1 | (1,1) | (4,3) | 1 < 4 | p += 1 | 0 |
| 3 | 1 | 1 | (4,2) | (4,3) | bằng | total += 2·3, p,q += 1 | 6 |
| — | 2 | 2 | hết A | | | dừng | **6** |

In [15]:
def trace_two_pointers(nums1, nums2):
    A = [(i, v) for i, v in enumerate(nums1) if v]
    B = [(i, v) for i, v in enumerate(nums2) if v]
    print("A =", A)
    print("B =", B)
    p = q = total = 0
    step = 0
    while p < len(A) and q < len(B):
        step += 1
        (i1, v1), (i2, v2) = A[p], B[q]
        if i1 == i2:
            total += v1 * v2
            act = f"bằng nhau -> total += {v1}*{v2}"
            p += 1
            q += 1
        elif i1 < i2:
            act = f"{i1} < {i2} -> p += 1"
            p += 1
        else:
            act = f"{i1} > {i2} -> q += 1"
            q += 1
        print(f"bước {step}: A[p]=({i1},{v1}) B[q]=({i2},{v2}) | {act:<26} | p={p} q={q} total={total}")
    print("kết quả:", total)
    return total

trace_two_pointers([0, 1, 0, 0, 2, 0, 0], [1, 0, 0, 0, 3, 0, 4])

A = [(1, 1), (4, 2)]
B = [(0, 1), (4, 3), (6, 4)]
bước 1: A[p]=(1,1) B[q]=(0,1) | 1 > 0 -> q += 1            | p=0 q=1 total=0
bước 2: A[p]=(1,1) B[q]=(4,3) | 1 < 4 -> p += 1            | p=1 q=1 total=0
bước 3: A[p]=(4,2) B[q]=(4,3) | bằng nhau -> total += 2*3  | p=2 q=2 total=6
kết quả: 6


6

In [16]:
def trace_dict(nums1, nums2):
    a = {i: v for i, v in enumerate(nums1) if v}
    b = {i: v for i, v in enumerate(nums2) if v}
    print("dict 1:", a, "| dict 2:", b)
    if len(b) < len(a):
        a, b = b, a
        print("hoán đổi để duyệt dict nhỏ hơn")
    total = 0
    for i, v in a.items():
        other = b.get(i, 0)
        total += v * other
        print(f"  i={i}: {v} * b.get({i}, 0)={other} -> total={total}")
    return total

trace_dict([1, 0, 0, 2, 3], [0, 3, 0, 4, 0])

dict 1: {0: 1, 3: 2, 4: 3} | dict 2: {1: 3, 3: 4}
hoán đổi để duyệt dict nhỏ hơn
  i=1: 3 * b.get(1, 0)=0 -> total=0
  i=3: 4 * b.get(3, 0)=2 -> total=8


8

## 9. Code nộp LeetCode

Đúng tên lớp/phương thức theo doocs: `SparseVector.__init__(self, nums)` và `dotProduct(self, vec)`.

In [17]:
class SparseVector:
    def __init__(self, nums: List[int]):
        self.d = {i: v for i, v in enumerate(nums) if v}

    # Return the dotProduct of two sparse vectors
    def dotProduct(self, vec: "SparseVector") -> int:
        a, b = self.d, vec.d
        if len(b) < len(a):
            a, b = b, a
        return sum(v * b.get(i, 0) for i, v in a.items())

# Cách LeetCode gọi:
for nums1, nums2, want in examples:
    v1 = SparseVector(nums1)
    v2 = SparseVector(nums2)
    ans = v1.dotProduct(v2)
    print(ans, "| mong đợi", want)

8 | mong đợi 8
0 | mong đợi 0
6 | mong đợi 6


## 10. Vì sao thuật toán đúng

**Cách 2 (dict):** Theo định nghĩa, `a·b = Σ a[i]b[i]`. Mọi số hạng có `a[i] = 0` hoặc `b[i] = 0` đều bằng 0 nên có thể bỏ. Các số hạng còn lại ứng với `i ∈ NZ(a) ∩ NZ(b)`. Khi duyệt mọi `i` trong dict nhỏ (giả sử `NZ(a)`), `b.get(i, 0)` bằng `b[i]` nếu `i ∈ NZ(b)` và bằng `0` nếu không — đúng giá trị thật của `b[i]`. Vậy tổng thu được chính xác là `Σ_{i∈NZ(a)} a[i]b[i] = a·b`. Hoán đổi vai trò không ảnh hưởng vì phép nhân giao hoán.

**Cách 3 (hai con trỏ):** Bất biến: *mọi cặp chỉ số chung có chỉ số nhỏ hơn `min(A[p].i, B[q].i)` đã được cộng đúng một lần.* Khi `A[p].i < B[q].i`, vì `B` tăng dần, mọi `B[q'].i` với `q' ≥ q` đều `≥ B[q].i > A[p].i`, nên `A[p].i` không có cặp → bỏ `p` an toàn. Trường hợp đối xứng tương tự. Khi bằng nhau thì cộng và tiến cả hai, mỗi chỉ số chung được cộng đúng một lần. Khi một danh sách hết, không còn chỉ số chung nào.

**Cách 4 (nhị phân):** với mỗi `i ∈ NZ(small)`, `bisect_left` tìm đúng vị trí của `i` trong `big.idx` nếu có; thu hẹp `lo` đúng vì `i` tăng dần.

## 11. Phân tích Time / Space

Gọi `n` = độ dài vector, `k₁, k₂` = số phần tử khác 0 của hai vector.

| Cách | Khởi tạo | `dotProduct` | Bộ nhớ mỗi vector |
|---|---|---|---|
| 1. Mảng đầy | `O(n)` | `O(n)` | `O(n)` |
| 2. Dict | `O(n)` | `O(min(k₁,k₂))` trung bình, xấu nhất `O(min·k)` khi va chạm hash | `O(k)` |
| 3. Cặp + hai con trỏ | `O(n)` | `O(k₁ + k₂)` xấu nhất | `O(k)` |
| 4. Cặp + nhị phân | `O(n)` | `O(min(k₁,k₂) · log max(k₁,k₂))` | `O(k)` |
| 4a. Thưa × dày | `O(n)` | `O(k_thưa)` | `O(n)` cho bên dày |

Khởi tạo luôn phải đọc hết `nums` nên `O(n)` là không tránh được. Lợi ích nằm ở chỗ: **một lần khởi tạo, nhiều lần nhân**.

## 12. Bộ test

### 12.1 Bộ chạy chuỗi lệnh giả lập (kiểu LeetCode)

**Đây là bản giả lập** (không phải bộ chấm của LeetCode). Định dạng lệnh tự đặt:
- `("new", nums)` → tạo vector mới, gán id = số thứ tự, trả `None`;
- `("dot", i, j)` → trả `vectors[i].dotProduct(vectors[j])`.

In [18]:
def run_commands(cls, commands):
    vectors, out = [], []
    for cmd in commands:
        if cmd[0] == "new":
            vectors.append(cls(cmd[1]))
            out.append(None)
        elif cmd[0] == "dot":
            out.append(vectors[cmd[1]].dotProduct(vectors[cmd[2]]))
        else:
            raise ValueError(cmd)
    return out

cmds = [("new", [1, 0, 0, 2, 3]), ("new", [0, 3, 0, 4, 0]), ("dot", 0, 1), ("dot", 1, 0), ("dot", 0, 0)]
print(run_commands(SparseVector, cmds))
print(run_commands(SparseVectorDense, cmds))

[None, None, 8, 8, 14]
[None, None, 8, 8, 14]


In [19]:
# 12.2 Ví dụ của đề + edge case
ALL = [SparseVectorDense, SparseVectorDict, SparseVectorPairs, SparseVectorHybrid, SparseVector]
cases = examples + [
    ([0], [0], 0),                      # n = 1, toàn 0
    ([5], [7], 35),                     # n = 1, khác 0
    ([0, 0, 0], [1, 2, 3], 0),          # một vector toàn 0
    ([100] * 6, [100] * 6, 60000),      # dày hoàn toàn, giá trị lớn nhất
    ([1, 0, 1, 0], [0, 1, 0, 1], 0),    # xen kẽ, không giao
    ([0, 0, 9], [0, 0, 9], 81),         # chỉ phần tử cuối
]
for cls in ALL:
    for a, b, want in cases:
        assert cls(a).dotProduct(cls(b)) == want, (cls.__name__, a, b)
        assert cls(b).dotProduct(cls(a)) == want, (cls.__name__, "giao hoán", a, b)
print("Tất cả", len(cases), "case x", len(ALL), "cài đặt: OK")

Tất cả 9 case x 5 cài đặt: OK


In [20]:
# 12.3 Chuỗi lệnh ngẫu nhiên, seed cố định, so với bản brute force (mảng đầy)
rng = random.Random(2024)
for trial in range(300):
    n = rng.randint(1, 30)
    density = rng.choice([0.0, 0.1, 0.3, 0.7, 1.0])
    cmds, count = [], 0
    for _ in range(rng.randint(1, 15)):
        if count == 0 or rng.random() < 0.4:
            nums = [rng.randint(1, 100) if rng.random() < density else 0 for _ in range(n)]
            cmds.append(("new", nums))
            count += 1
        else:
            cmds.append(("dot", rng.randrange(count), rng.randrange(count)))
    want = run_commands(SparseVectorDense, cmds)
    for cls in ALL[1:]:
        assert run_commands(cls, cmds) == want, (cls.__name__, cmds)
print("300 chuỗi lệnh ngẫu nhiên: mọi cài đặt khớp baseline")

300 chuỗi lệnh ngẫu nhiên: mọi cài đặt khớp baseline


In [21]:
# 12.4 Test lớn: n = 10^5, đúng ràng buộc, so sánh các cách
rng = random.Random(7)
for k1, k2 in [(10, 50000), (1000, 1000), (100000, 100000), (0, 500)]:
    a, b = random_sparse(10**5, k1, rng), random_sparse(10**5, k2, rng)
    want = sum(x * y for x, y in zip(a, b))
    res = [cls(a).dotProduct(cls(b)) for cls in ALL[1:]]
    assert all(r == want for r in res), (k1, k2, res, want)
    print(f"k1={k1:<6} k2={k2:<6} -> {want}")
print("Test lớn: OK")

k1=10     k2=50000  -> 6178


k1=1000   k2=1000   -> 23030


k1=100000 k2=100000 -> 255119211
k1=0      k2=500    -> 0
Test lớn: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ lưu các giá trị khác 0, **quên chỉ số**

Nén `[1,0,0,2,3]` thành `[1,2,3]` rồi nhân theo thứ tự → mất thông tin vị trí.

In [22]:
class WrongValuesOnly:
    def __init__(self, nums):
        self.vals = [v for v in nums if v]          # SAI: bỏ mất chỉ số

    def dotProduct(self, vec):
        return sum(x * y for x, y in zip(self.vals, vec.vals))

a, b = [1, 0, 0, 2, 3], [0, 3, 0, 4, 0]
print("sai:", WrongValuesOnly(a).dotProduct(WrongValuesOnly(b)), "| đúng:", SparseVector(a).dotProduct(SparseVector(b)))
# Sửa: lưu cả chỉ số, ví dụ dict {i: v} hoặc list cặp (i, v)

sai: 11 | đúng: 8


### Lỗi 2 — Hai con trỏ: tiến **cả hai** khi chỉ số khác nhau

Khi `i1 ≠ i2` chỉ được tiến con trỏ có chỉ số **nhỏ hơn**. Tiến cả hai sẽ bỏ lỡ cặp trùng phía sau.

In [23]:
def wrong_two_pointers(A, B):
    p = q = total = 0
    while p < len(A) and q < len(B):
        if A[p][0] == B[q][0]:
            total += A[p][1] * B[q][1]
        p += 1                                   # SAI: luôn tiến cả hai
        q += 1
    return total

def right_two_pointers(A, B):
    p = q = total = 0
    while p < len(A) and q < len(B):
        if A[p][0] == B[q][0]:
            total += A[p][1] * B[q][1]
            p += 1
            q += 1
        elif A[p][0] < B[q][0]:
            p += 1
        else:
            q += 1
    return total

A = [(0, 1), (4, 2)]      # vector [1,0,0,0,2]
B = [(4, 3)]              # vector [0,0,0,0,3]
print("sai:", wrong_two_pointers(A, B), "| đúng:", right_two_pointers(A, B))
# Bản sai so (0,1) với (4,3), không khớp nhưng lại tiến cả q -> B hết, bỏ lỡ cặp chỉ số 4.

sai: 0 | đúng: 6


### Lỗi 3 — Dùng `b[i]` thay vì `b.get(i, 0)`

Chỉ số có trong dict nhỏ nhưng không có trong dict lớn → `KeyError`. Ở đây ta bắt lỗi để in ra, không để notebook dừng.

In [24]:
class WrongKeyError:
    def __init__(self, nums):
        self.d = {i: v for i, v in enumerate(nums) if v}

    def dotProduct(self, vec):
        return sum(v * vec.d[i] for i, v in self.d.items())   # SAI: vec.d[i] có thể không tồn tại

try:
    print(WrongKeyError([1, 0, 0, 2, 3]).dotProduct(WrongKeyError([0, 3, 0, 4, 0])))
except KeyError as e:
    print("Lỗi KeyError với khoá", e, "-> sửa bằng vec.d.get(i, 0) hoặc kiểm tra 'i in vec.d'")
print("đúng:", SparseVector([1, 0, 0, 2, 3]).dotProduct(SparseVector([0, 3, 0, 4, 0])))

Lỗi KeyError với khoá 0 -> sửa bằng vec.d.get(i, 0) hoặc kiểm tra 'i in vec.d'
đúng: 8


### Lỗi 4 — Đảo khoá/giá trị: lưu `{v: i}`

Nhiều người viết nhầm `{v: i for i, v in ...}`. Khi có giá trị trùng nhau, khoá bị ghi đè và phép tra theo chỉ số trở nên vô nghĩa.

In [25]:
class WrongSwapped:
    def __init__(self, nums):
        self.d = {v: i for i, v in enumerate(nums) if v}        # SAI: khoá là giá trị

    def dotProduct(self, vec):
        return sum(k * vec.d.get(k, 0) for k in self.d)          # ý định "tra theo chỉ số" bị lệch

a, b = [2, 0, 2], [2, 2, 2]          # giá trị 2 lặp lại -> khoá bị ghi đè
print("WrongSwapped.d =", WrongSwapped(a).d)
print("sai:", WrongSwapped(a).dotProduct(WrongSwapped(b)), "| đúng:", SparseVector(a).dotProduct(SparseVector(b)))

WrongSwapped.d = {2: 2}
sai: 4 | đúng: 8


### Lỗi 5 — Dừng tìm nhị phân sai: so `pos` mà không kiểm tra `big.idx[pos] == i`

`bisect_left` trả vị trí **chèn**, không đảm bảo phần tử ở đó bằng `i`.

In [26]:
def wrong_bisect(small_idx, small_val, big_idx, big_val):
    total = 0
    for i, v in zip(small_idx, small_val):
        pos = bisect_left(big_idx, i)
        if pos < len(big_idx):                    # SAI: thiếu kiểm tra big_idx[pos] == i
            total += v * big_val[pos]
    return total

a, b = [0, 1, 0, 0, 2, 0, 0], [1, 0, 0, 0, 3, 0, 4]
sa, sb = SparseVectorHybrid(a), SparseVectorHybrid(b)
print("sai:", wrong_bisect(sa.idx, sa.val, sb.idx, sb.val), "| đúng:", sa.dotProduct(sb))

sai: 9 | đúng: 6


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **Nghe "sparse" → nghĩ ngay "chỉ lưu phần khác 0"** (dict hoặc list cặp). Đây là ý tưởng của định dạng COO/CSR trong thư viện ma trận thưa (scipy.sparse).
- **Tích chỉ khác 0 khi cả hai khác 0** → bài toán quy về **giao hai tập chỉ số**. Giao hai tập: dict (hash) hoặc hai con trỏ (đã sắp) hoặc nhị phân (chênh lệch kích thước).
- **Luôn duyệt bên nhỏ hơn** khi dùng hash hoặc nhị phân.
- Trong phỏng vấn, trình bày theo thứ tự: mảng đầy → dict → hai con trỏ (không cần hash) → nhị phân (follow-up chênh lệch kích thước). Nói rõ trade-off: hash có hằng số lớn, hai con trỏ ổn định, nhị phân tốt khi `k₁ ≪ k₂`.
- `enumerate` + `if v` tạo danh sách đã **tự sắp theo chỉ số** — không cần sort.

## 15. Biến thể của bài

1. **LC 311 — Sparse Matrix Multiplication:** nhân hai ma trận thưa; mỗi phần tử kết quả là một tích vô hướng hàng × cột. Lưu mỗi hàng của A dạng cặp, bỏ qua `A[i][k] = 0`.
2. **Cộng/trừ hai vector thưa:** trộn hai danh sách cặp bằng hai con trỏ (giống merge của merge sort), bỏ kết quả bằng 0.
3. **Cosine similarity giữa văn bản:** vector tần suất từ rất thưa → `cos = a·b / (|a|·|b|)`.
4. **Dữ liệu dạng run-length** (ví dụ LC 1868 — Product of Two Run-Length Encoded Arrays): hai con trỏ trên các khối `(giá trị, độ dài)`.
5. **Giá trị âm / số thực:** thuật toán không đổi; chỉ cần cẩn thận với so sánh `== 0` cho số thực (dùng ngưỡng nhỏ).
6. **Giao hai mảng đã sắp (LC 349/350):** cùng khung hai con trỏ.

## 16. So sánh các cách

| Cách | Ý tưởng | Time `dotProduct` | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Mảng đầy | nhân từng vị trí | `O(n)` | `O(n)` | vector dày, hoặc làm oracle kiểm thử |
| 2. Dict `i → v` | duyệt dict nhỏ, tra dict lớn | `O(min(k₁,k₂))` trung bình | `O(k)` | code ngắn nhất, đủ cho LeetCode |
| 3. Cặp + hai con trỏ | trộn hai danh sách đã sắp | `O(k₁+k₂)` xấu nhất | `O(k)` | muốn tránh hash, phỏng vấn hỏi "không dùng hash" |
| 4. Cặp + nhị phân | duyệt bên nhỏ, tìm nhị phân bên lớn | `O(k_min log k_max)` | `O(k)` | follow-up: kích thước chênh lệch lớn |
| 4a. Thưa × dày | duyệt bên thưa, tra `dense[i]` | `O(k_thưa)` | `O(n)` | follow-up: chỉ một vector thưa |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** Với `nums1 = [0, 4, 0, 5, 0, 6]`, `nums2 = [2, 3, 0, 0, 1, 2]`, `SparseVector(nums1).dotProduct(SparseVector(nums2))` trả về bao nhiêu? Dict của mỗi vector là gì?

**Bài 2 (tính tay):** Với `A = [(0,1), (2,5), (7,3), (9,1)]` và `B = [(2,2), (3,4), (9,9)]`, thuật toán hai con trỏ thực hiện **bao nhiêu vòng lặp** và trả về bao nhiêu? Viết hàm `ex2_count_steps(A, B)` trả về `(số vòng lặp, tổng)` để kiểm tra.

**Bài 3 (điền TODO):** Viết `ex3_add_sparse(A, B)` cộng hai vector thưa dạng list cặp đã sắp, trả về list cặp đã sắp, **bỏ các vị trí có tổng bằng 0** (giá trị có thể âm).

**Bài 4 (tự viết lại):** Viết lại lớp `Ex4SparseVector` dùng **hai con trỏ**, nhưng lưu chỉ số và giá trị ở **hai list riêng** `self.idx`, `self.val`.

**Bài 5 (mini challenge — LC 311):** Viết `ex5_sparse_matmul(A, B)` nhân hai ma trận thưa `A (m×k)` và `B (k×n)`, bỏ qua các phần tử 0 để nhanh hơn cách ba vòng lặp ngây thơ.

In [27]:
# Bài 3 — khung TODO (chỉ định nghĩa, KHÔNG gọi)
def ex3_add_sparse(A, B):
    """A, B: list cặp (chỉ số, giá trị) đã sắp theo chỉ số. Trả về list cặp của A + B."""
    res = []
    p = q = 0
    # TODO: trộn hai danh sách bằng hai con trỏ
    # TODO: khi chỉ số bằng nhau, cộng giá trị; chỉ thêm vào res nếu tổng khác 0
    # TODO: đừng quên phần còn lại của A hoặc B sau vòng lặp
    return res

In [28]:
# Bài 4 — khung TODO (chỉ định nghĩa, KHÔNG gọi)
class Ex4SparseVector:
    def __init__(self, nums):
        # TODO: self.idx = các chỉ số khác 0; self.val = giá trị tương ứng
        pass

    def dotProduct(self, vec):
        # TODO: hai con trỏ p trên self, q trên vec
        pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Viết dict của từng vector. (2) Tìm giao chỉ số. (3) Giao là `{1, 5}` → cộng hai tích.

**Bài 2.** (1) Mỗi vòng lặp tiến ít nhất một con trỏ. (2) Ghi bảng `p, q` từng bước. (3) Vòng lặp dừng khi `B` hết sau khi khớp chỉ số 9.

**Bài 3.** (1) Khung giống hai con trỏ của dot product, nhưng khi chỉ số khác nhau thì **thêm** phần tử nhỏ hơn vào kết quả. (2) Bằng nhau: `s = v1 + v2`, chỉ thêm khi `s != 0`. (3) Sau vòng lặp: `res.extend(A[p:])` và `res.extend(B[q:])`.

**Bài 4.** (1) `self.idx = [i for i, v in enumerate(nums) if v]`. (2) `self.val = [nums[i] for i in self.idx]`. (3) So sánh `self.idx[p]` với `vec.idx[q]`.

**Bài 5.** (1) `C[i][j] = Σ_t A[i][t]·B[t][j]`. (2) Duyệt `i, t` với `A[i][t] ≠ 0`, rồi duyệt các `(j, b)` khác 0 của hàng `t` trong `B`. (3) Tiền xử lý mỗi hàng của `B` thành list cặp khác 0.

## 19. Đáp án tham khảo cho bài tập

In [29]:
# Bài 1
n1, n2 = [0, 4, 0, 5, 0, 6], [2, 3, 0, 0, 1, 2]
ans1 = SparseVector(n1).dotProduct(SparseVector(n2))
print("dict1:", SparseVector(n1).d, "| dict2:", SparseVector(n2).d, "| tích:", ans1)
assert ans1 == 4 * 3 + 6 * 2 == 24
print("Bài 1: OK")

dict1: {1: 4, 3: 5, 5: 6} | dict2: {0: 2, 1: 3, 4: 1, 5: 2} | tích: 24
Bài 1: OK


In [30]:
# Bài 2
def ex2_count_steps(A, B):
    p = q = total = steps = 0
    while p < len(A) and q < len(B):
        steps += 1
        if A[p][0] == B[q][0]:
            total += A[p][1] * B[q][1]
            p += 1
            q += 1
        elif A[p][0] < B[q][0]:
            p += 1
        else:
            q += 1
    return steps, total

res2 = ex2_count_steps([(0, 1), (2, 5), (7, 3), (9, 1)], [(2, 2), (3, 4), (9, 9)])
print("(số vòng lặp, tổng) =", res2)
assert res2 == (5, 5 * 2 + 1 * 9)
print("Bài 2: OK")

(số vòng lặp, tổng) = (5, 19)
Bài 2: OK


In [31]:
# Bài 3
def ex3_add_sparse_sol(A, B):
    res, p, q = [], 0, 0
    while p < len(A) and q < len(B):
        (i1, v1), (i2, v2) = A[p], B[q]
        if i1 == i2:
            if v1 + v2 != 0:
                res.append((i1, v1 + v2))
            p += 1
            q += 1
        elif i1 < i2:
            res.append(A[p])
            p += 1
        else:
            res.append(B[q])
            q += 1
    res.extend(A[p:])
    res.extend(B[q:])
    return res

rng3 = random.Random(3)
for _ in range(500):
    n = rng3.randint(1, 15)
    a = [rng3.choice([0, 0, 0, -2, -1, 1, 2, 5]) for _ in range(n)]
    b = [rng3.choice([0, 0, 0, -2, -1, 1, 2, 5]) for _ in range(n)]
    A = [(i, v) for i, v in enumerate(a) if v]
    B = [(i, v) for i, v in enumerate(b) if v]
    want = [(i, x + y) for i, (x, y) in enumerate(zip(a, b)) if x + y != 0]
    assert ex3_add_sparse_sol(A, B) == want
print("Bài 3: OK, ví dụ:", ex3_add_sparse_sol([(0, 1), (2, 3)], [(2, -3), (4, 7)]))

Bài 3: OK, ví dụ: [(0, 1), (4, 7)]


In [32]:
# Bài 4
class Ex4SparseVectorSol:
    def __init__(self, nums):
        self.idx = [i for i, v in enumerate(nums) if v]
        self.val = [nums[i] for i in self.idx]

    def dotProduct(self, vec):
        p = q = total = 0
        while p < len(self.idx) and q < len(vec.idx):
            if self.idx[p] == vec.idx[q]:
                total += self.val[p] * vec.val[q]
                p += 1
                q += 1
            elif self.idx[p] < vec.idx[q]:
                p += 1
            else:
                q += 1
        return total

rng4 = random.Random(4)
for _ in range(400):
    n = rng4.randint(1, 25)
    a = [rng4.randint(1, 100) if rng4.random() < 0.3 else 0 for _ in range(n)]
    b = [rng4.randint(1, 100) if rng4.random() < 0.3 else 0 for _ in range(n)]
    assert Ex4SparseVectorSol(a).dotProduct(Ex4SparseVectorSol(b)) == SparseVectorDense(a).dotProduct(SparseVectorDense(b))
print("Bài 4: OK")

Bài 4: OK


In [33]:
# Bài 5
def ex5_sparse_matmul(A, B):
    m, k, n = len(A), len(B), len(B[0])
    rowsB = [[(j, B[t][j]) for j in range(n) if B[t][j]] for t in range(k)]
    C = [[0] * n for _ in range(m)]
    for i in range(m):
        for t in range(k):
            a = A[i][t]
            if a:
                for j, b in rowsB[t]:
                    C[i][j] += a * b
    return C

def naive_matmul(A, B):
    return [[sum(A[i][t] * B[t][j] for t in range(len(B))) for j in range(len(B[0]))] for i in range(len(A))]

print(ex5_sparse_matmul([[1, 0, 0], [-1, 0, 3]], [[7, 0, 0], [0, 0, 0], [0, 0, 1]]))
assert ex5_sparse_matmul([[1, 0, 0], [-1, 0, 3]], [[7, 0, 0], [0, 0, 0], [0, 0, 1]]) == [[7, 0, 0], [-7, 0, 3]]
rng5 = random.Random(5)
for _ in range(200):
    m, k, n = rng5.randint(1, 6), rng5.randint(1, 6), rng5.randint(1, 6)
    A = [[rng5.choice([0, 0, 0, 1, -2, 3]) for _ in range(k)] for _ in range(m)]
    B = [[rng5.choice([0, 0, 0, 2, -1, 4]) for _ in range(n)] for _ in range(k)]
    assert ex5_sparse_matmul(A, B) == naive_matmul(A, B)
print("Bài 5: OK")

[[7, 0, 0], [-7, 0, 3]]
Bài 5: OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
class SparseVector:
    def __init__(self, nums):
        self.d = {i: v for i, v in enumerate(nums) if v}     # chỉ lưu phần khác 0

    def dotProduct(self, vec):
        a, b = self.d, vec.d
        if len(b) < len(a):                                   # duyệt bên nhỏ
            a, b = b, a
        return sum(v * b.get(i, 0) for i, v in a.items())

# Không dùng hash: list cặp (i, v) + hai con trỏ (tiến con trỏ có chỉ số nhỏ hơn)
# Follow-up chênh lệch: duyệt bên nhỏ + bisect_left bên lớn
```

**Checklist trước khi submit:**
- [ ] Tên lớp `SparseVector`, phương thức `dotProduct(self, vec)` đúng chính tả (chữ `P` hoa).
- [ ] Lưu **cả chỉ số lẫn giá trị**, chỉ cho phần tử khác 0.
- [ ] Dùng `get(i, 0)` (hoặc `in`) — không dùng `b[i]` trần.
- [ ] Hai con trỏ: chỉ tiến con trỏ có chỉ số nhỏ hơn khi khác nhau; tiến cả hai khi bằng nhau.
- [ ] Duyệt vector có ít phần tử khác 0 hơn.
- [ ] Thử nhanh: ba ví dụ → `8, 0, 6`; vector toàn 0 → `0`; `[5]·[7] = 35`.
- [ ] Trả lời được follow-up: một vector dày → tra `dense[i]`; chênh lệch kích thước → tìm nhị phân.